# run 2: confirmation, H2 and H3
Model: Qwen3-4B-Base on a **T4 GPU**. Reads from and saves to **My Drive / see-vs-do** (it needs run 1's `results/` folder there). Use **Runtime → Run all**.

| Stage | Question | Time on a T4 |
|---|---|---|
| E | **Confirmation:** does run 1's pattern hold on 25 new scenarios the method has never seen? | ~40 min |
| F | **H2:** are edits seeing-like *because* the mediator direction overlaps the cause direction? Remove the overlap and edit again. | ~20 min |
| G | **H3:** colliders. Does an edit to C act like seeing C (B rises, and stating A then lowers B), or like doing C? | ~25 min |

All settings below are fixed **before** these results exist. The edit strength comes from run 1 by the rule in the settings cell.

In [ ]:
# Mount Google Drive and work inside the see-vs-do folder (story files live there; results are saved there)
import os
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/see-vs-do")
OUT = "results"
os.makedirs(OUT, exist_ok=True)
assert os.path.isdir("results"), "run 1 results not found"
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # run 1 (exploration) held-out variants
EVAL_CONFIRM = range(16, 24)    # run 2 uses all 8 held-out variants
LAYER_STEP = 4                  # plan of record: every 4th layer
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

In [ ]:
# ---------- Shared helpers for this run
import glob, matplotlib.pyplot as plt
PROBES = {"cause": "q_cause", "effect": "q_eff", "med": "q_med"}
rng = np.random.default_rng(0)

def load_items(path):
    x = pd.read_csv(path)
    x["variant"] = x.item_id.str.rsplit("_", n=1).str[1].astype(int)
    for c in [c for c in x.columns if c.startswith("span_")]:
        x[c] = x[c].apply(ast.literal_eval)
    return x

def reference(evdf, probes, conds):
    r = evdf[["scenario_id", "item_id"]].copy()
    for cond in conds:
        for p, q in probes.items():
            r[f"{p}_{cond}"] = belief(evdf[cond].tolist(), evdf[q].tolist(), evdf[f"span_{cond}"].tolist())
    return r

def scenario_table(ref, cause="cause", effect="effect", base="base"):
    """Per-scenario mean shifts vs base, ratio r = dcause/deffect, and the fixed filters."""
    t = pd.DataFrame(index=sorted(ref.scenario_id.unique()))
    for cond in ["see", "do"]:
        for p in [cause, effect]:
            t[f"d{p}_{cond}"] = (ref[f"{p}_{cond}"] - ref[f"{p}_{base}"]).groupby(ref.scenario_id).mean()
    t["r_see"] = t[f"d{cause}_see"] / t[f"d{effect}_see"]
    t["r_do"] = t[f"d{cause}_do"] / t[f"d{effect}_do"]
    t["valid"] = ((t[f"d{cause}_see"] > 0.5) & (t[f"d{effect}_see"] > 0.5) & (t[f"d{effect}_do"] > 0.5)
                  & (t.r_see - t.r_do > 0.25))
    return t

def directions(trdf, true_col, false_col, q_col, scen):
    Ht = key_states(trdf[true_col].tolist(), trdf[q_col].tolist(), trdf[f"span_{true_col}"].tolist(), LAYERS)
    Hf = key_states(trdf[false_col].tolist(), trdf[q_col].tolist(), trdf[f"span_{false_col}"].tolist(), LAYERS)
    return {s: Ht[(trdf.scenario_id == s).values].mean(0) - Hf[(trdf.scenario_id == s).values].mean(0) for s in scen}

def rand_like(dirs, seed):
    g = torch.Generator().manual_seed(seed)
    out = {}
    for s, d in dirs.items():
        r = torch.randn(d.shape, generator=g)
        out[s] = r / r.norm(dim=-1, keepdim=True) * d.norm(dim=-1, keepdim=True)
    return out

def run_sweep(evdf, text_col, dirmaps, settings, probes, tag, positions=POSITIONS):
    """settings: list of (kind, alpha); dirmaps[kind][scenario] = [L, H]. Saves one CSV per layer; skips finished layers."""
    stories, spans = evdf[text_col].tolist(), evdf[f"span_{text_col}"].tolist()
    base_b = {p: belief(stories, evdf[q].tolist(), spans) for p, q in probes.items()}
    t0 = time.time()
    for li, L in enumerate(LAYERS):
        path = f"{OUT}/{tag}_layer{L:02d}.csv"
        if os.path.exists(path):
            continue
        rows = []
        for mode in positions:
            for kind, alpha in settings:
                vecs = torch.stack([alpha * dirmaps[kind][s][li] for s in evdf.scenario_id])
                r = evdf[["scenario_id", "item_id"]].copy()
                r["layer"], r["position"], r["kind"], r["alpha"], r["text"] = L, mode, kind, alpha, text_col
                for p, q in probes.items():
                    b = belief(stories, evdf[q].tolist(), spans, edit=dict(layer=L, vecs=vecs, mode=mode))
                    r[f"d{p}"], r[f"raw_{p}"] = b - base_b[p], b
                rows.append(r)
        pd.concat(rows).to_csv(path, index=False)
        print(f"  {tag}: layer {L} done ({(time.time() - t0) / 60:.1f} min)")
    return pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/{tag}_layer*.csv")))

def score(sw, table, cause="cause", effect="effect"):
    g = sw.groupby(["scenario_id", "layer", "position", "kind", "alpha"])[[f"d{cause}", f"d{effect}"]].mean().reset_index()
    g = g.merge(table[[f"d{effect}_see", "r_see", "r_do"]], left_on="scenario_id", right_index=True)
    g["works"] = g[f"d{effect}"] >= 0.5 * g[f"d{effect}_see"]
    g["r_edit"] = g[f"d{cause}"] / g[f"d{effect}"]
    g["D"] = np.where(g.works, (g.r_see - g.r_edit) / (g.r_see - g.r_do), np.nan)
    g["D_clip"] = g.D.clip(0, 1)
    return g

def boot_median(x, n=2000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

def summarise(g):
    out = []
    for k, grp in g.groupby(["kind", "alpha", "layer", "position"]):
        lo, hi = boot_median(grp.D)
        out.append(dict(zip(["kind", "alpha", "layer", "position"], k), n_scen=len(grp), works_rate=grp.works.mean(),
                        n_scored=int(grp.D.notna().sum()), D_median=grp.D.median(), D_clip_median=grp.D_clip.median(),
                        ci_lo=lo, ci_hi=hi))
    return pd.DataFrame(out)

def heatmap(ax, summ, title, min_n=5):
    m = summ.pivot(index="layer", columns="position", values="D_clip_median").reindex(columns=POSITIONS)
    n = summ.pivot(index="layer", columns="position", values="n_scored").reindex(columns=POSITIONS)
    m = m.where(n >= min_n)
    im = ax.imshow(m.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v, k = m.values[i, j], n.values[i, j]
            ax.text(j, i, f"n={int(k) if not np.isnan(k) else 0}" if np.isnan(v) else f"{v:.2f}\nn={int(k)}",
                    ha="center", va="center", fontsize=8, color="white" if (not np.isnan(v) and v > 0.6) else "black")
    ax.set_xticks(range(len(POSITIONS)), POSITIONS); ax.set_yticks(range(len(m.index)), m.index)
    ax.set_title(title); ax.set_xlabel("edit position")
    return im

## Settings, fixed before any run-2 result exists
**Edit strength α\*:** from run 1's `D_summary.csv`, the α (real direction) at which edits most often pass the matched-effect rule, averaged over all layers and positions; ties go to the smaller α. Run 2 uses only α\*, at every layer and position.

In [ ]:
s1 = pd.read_csv(f"{OUT}/D_summary.csv")
wr = s1[s1.kind == "real"].groupby("alpha").works_rate.mean()
display(wr.round(3).to_frame("run-1 works rate"))
ALPHA_STAR = float(wr[wr == wr.max()].index.min())
print("ALPHA_STAR =", ALPHA_STAR)
run1 = s1[(s1.kind == "real") & (s1.alpha == ALPHA_STAR)].copy()

## Stage E: confirmation on 25 new scenarios
Same pipeline as run 1, on scenarios never used before: reference, filters, directions, then the sweep (real, random and unrelated directions, all at α\*).
**Replication test:** across the 27 layer × position cells, does the confirmation heatmap match the run-1 heatmap? Spearman correlation of the cell medians, and the share of cells where both runs fall on the same side of D = 0.5.

In [ ]:
cf = load_items("stories_chain_confirm.csv")
cf_tr = cf[cf.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
cf_ev = cf[cf.variant.isin(EVAL_CONFIRM)].reset_index(drop=True)
CF_SCEN = sorted(cf.scenario_id.unique())

path = f"{OUT}/E_reference_items.csv"
cf_ref = pd.read_csv(path) if os.path.exists(path) else reference(cf_ev, PROBES, ["base", "see", "do"])
cf_ref.to_csv(path, index=False)
cf_sc = scenario_table(cf_ref)
cf_sc.to_csv(f"{OUT}/E_reference_scenarios.csv")
display(cf_sc.round(2))
CF_VALID = cf_sc.index[cf_sc.valid].tolist()
print(f"valid confirmation scenarios: {len(CF_VALID)} / {len(cf_sc)}")

CF_DIRS = directions(cf_tr, "see", "neg", "q_med", CF_SCEN)
torch.save({"layers": LAYERS, "dirs": CF_DIRS}, f"{OUT}/E_directions.pt")
pool = CF_VALID if len(CF_VALID) > 1 else CF_SCEN
cf_maps = {"real": CF_DIRS, "random": rand_like(CF_DIRS, 1),
           "unrelated": {s: CF_DIRS[pool[(pool.index(s) + 1) % len(pool)]] / CF_DIRS[pool[(pool.index(s) + 1) % len(pool)]].norm(dim=-1, keepdim=True) * CF_DIRS[s].norm(dim=-1, keepdim=True) for s in pool}}
CF_EV = cf_ev[cf_ev.scenario_id.isin(CF_VALID)].reset_index(drop=True)
print(f"{len(CF_EV)} items x {len(LAYERS) * len(POSITIONS) * 3 * len(PROBES)} settings")
cf_sw = run_sweep(CF_EV, "base", cf_maps, [("real", ALPHA_STAR), ("random", ALPHA_STAR), ("unrelated", ALPHA_STAR)], PROBES, "E_sweep")
cf_g = score(cf_sw, cf_sc); cf_g.to_csv(f"{OUT}/E_scenario_scores.csv", index=False)
cf_summ = summarise(cf_g); cf_summ.to_csv(f"{OUT}/E_summary.csv", index=False)

display(cf_summ.groupby("kind")[["works_rate", "D_median"]].mean().round(2))
from scipy.stats import spearmanr
both = run1.merge(cf_summ[cf_summ.kind == "real"], on=["layer", "position"], suffixes=("_run1", "_confirm"))
ok = both.dropna(subset=["D_clip_median_run1", "D_clip_median_confirm"])
ok = ok[(ok.n_scored_run1 >= 5) & (ok.n_scored_confirm >= 5)]
rho, p = spearmanr(ok.D_clip_median_run1, ok.D_clip_median_confirm) if len(ok) > 2 else (np.nan, np.nan)
same = ((ok.D_clip_median_run1 > 0.5) == (ok.D_clip_median_confirm > 0.5)).mean() if len(ok) else np.nan
print(f"Replication over {len(ok)} cells with n >= 5 in both runs: Spearman rho = {rho:.2f} (p = {p:.3f}); same side of 0.5 in {same:.0%} of cells")
both.to_csv(f"{OUT}/E_replication_cells.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
heatmap(axes[0], run1, f"Run 1: 17 exploration scenarios (alpha={ALPHA_STAR})")
im = heatmap(axes[1], cf_summ[cf_summ.kind == "real"], f"Run 2: {len(CF_VALID)} new scenarios")
axes[0].set_ylabel("layer"); fig.colorbar(im, ax=axes, label="median D (0 = seeing, 1 = doing)")
plt.savefig(f"{OUT}/E_replication_heatmaps.png", dpi=150, bbox_inches="tight"); plt.show()

## Stage F: H2, is seeing caused by overlap with the cause direction?
1. **Cause direction:** mean activation when the cause is stated true minus false, at the last token of the cause sentence.
2. **Correlation:** for each scenario and layer, cosine similarity between the mediator and cause directions, against that scenario's D.
3. **Intervention:** remove the cause component, B⊥ = B − (B·Â)Â, rescale to B's length, and edit again. H2 predicts D rises (more doing-like). Paired comparison per scenario, with a bootstrap CI.
Caveat: this removes only the *linear* overlap at that layer, so a null result is weaker evidence than a positive one.

In [ ]:
ex = d   # run-1 (exploration) stories, loaded in the setup cell
ex_tr = ex[ex.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ex_sc = pd.read_csv(f"{OUT}/A_reference_scenarios.csv", index_col=0)
EX_VALID = ex_sc.index[ex_sc.valid].tolist()
EX_DIRS = torch.load(f"{OUT}/B_directions.pt")["dirs"]

sets = {"exploration": (ex_tr, EX_DIRS, EX_VALID, ex_sc, ev), "confirmation": (cf_tr, CF_DIRS, CF_VALID, cf_sc, cf_ev)}
f_rows, f_g = [], []
for name, (tr, mdirs, valid, table, evd) in sets.items():
    cdirs = directions(tr, "cause_t", "cause_f", "q_cause", valid)
    proj = {}
    for s in valid:
        B, A = mdirs[s], cdirs[s]
        Ahat = A / A.norm(dim=-1, keepdim=True)
        Bp = B - (B * Ahat).sum(-1, keepdim=True) * Ahat
        proj[s] = Bp / Bp.norm(dim=-1, keepdim=True) * B.norm(dim=-1, keepdim=True)
        for li, L in enumerate(LAYERS):
            f_rows.append(dict(set=name, scenario_id=s, layer=L, cos=torch.nn.functional.cosine_similarity(B[li], A[li], dim=0).item()))
    EVs = evd[evd.scenario_id.isin(valid)].reset_index(drop=True)
    if len(EVs) == 0:
        print(f"no valid scenarios in {name}, skipping"); continue
    sw = run_sweep(EVs, "base", {"projected": proj, "real": mdirs}, [("projected", ALPHA_STAR)], PROBES, f"F_{name}")
    g = score(sw, table); g["set"] = name
    f_g.append(g)
cos = pd.DataFrame(f_rows); cos.to_csv(f"{OUT}/F_cosines.csv", index=False)
gp = pd.concat(f_g)
greal = pd.concat([pd.read_csv(f"{OUT}/D_scenario_scores.csv").query("kind == 'real' and alpha == @ALPHA_STAR").assign(set="exploration"),
                   cf_g.query("kind == 'real'").assign(set="confirmation")])
pair = greal.merge(gp, on=["set", "scenario_id", "layer", "position"], suffixes=("_real", "_proj"))
pair = pair.merge(cos, on=["set", "scenario_id", "layer"])
pair.to_csv(f"{OUT}/F_paired.csv", index=False)

display(cos.groupby("layer").cos.agg(["mean", "std"]).round(3))
res = []
for (L, pos), grp in pair.groupby(["layer", "position"]):
    both_ok = grp.dropna(subset=["D_real", "D_proj"])
    diff = both_ok.D_clip_proj - both_ok.D_clip_real
    lo, hi = boot_median(diff)
    from scipy.stats import spearmanr as _sp
    rc = _sp(grp.cos, grp.D_clip_real, nan_policy="omit")[0] if grp.D_real.notna().sum() > 4 else np.nan
    res.append(dict(layer=L, position=pos, n_pairs=len(both_ok), median_shift=diff.median(), ci_lo=lo, ci_hi=hi,
                    spearman_cos_vs_D=rc, works_real=grp.works_real.mean(), works_proj=grp.works_proj.mean()))
f_res = pd.DataFrame(res); f_res.to_csv(f"{OUT}/F_summary.csv", index=False)
display(f_res.round(2))
print("H2 predicts median_shift > 0 (removing the overlap makes edits more doing-like) and a negative spearman_cos_vs_D.")

## Stage G: H3, collider edits
Rule: C happens if A or B. Edit **C** into the neutral story (no words changed) and ask about **B**.
* **D for colliders:** r = ΔB / ΔC (the C question is the manipulation check), D = (r_see − r_do... same formula and filters as chains). Seeing C raises B (D near 0); doing C does not (D near 1).
* **Explaining away under the edit:** does stating A lower B *more* when C has been edited in? Interaction = [B(C) − B(C, A)] − [B(no C) − B(A only)]. The same interaction is computed from the text versions. Seeing-like edits should show a positive interaction like the text; doing-like edits should show about zero.

In [ ]:
co = load_items("stories_collider24.csv")
co_tr = co[co.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
co_ev = co[co.variant.isin(EVAL_CONFIRM)].reset_index(drop=True)
CO_SCEN = sorted(co.scenario_id.unique())
CP = {"b": "q_b", "c": "q_c"}
path = f"{OUT}/G_reference_items.csv"
co_ref = pd.read_csv(path) if os.path.exists(path) else reference(co_ev, CP, ["base", "see", "see_A", "A_only", "do"])
co_ref.to_csv(path, index=False)
co_sc = scenario_table(co_ref, cause="b", effect="c")
co_sc["EA_interaction_text"] = ((co_ref.b_see - co_ref.b_see_A) - (co_ref.b_base - co_ref.b_A_only)).groupby(co_ref.scenario_id).mean()
co_sc.to_csv(f"{OUT}/G_reference_scenarios.csv")
display(co_sc.round(2))
CO_VALID = co_sc.index[co_sc.valid].tolist()
print(f"valid collider scenarios: {len(CO_VALID)} / {len(co_sc)}")

CO_DIRS = directions(co_tr, "see", "neg", "q_c", CO_SCEN)
co_maps = {"real": CO_DIRS, "random": rand_like(CO_DIRS, 2)}
CO_EV = co_ev[co_ev.scenario_id.isin(CO_VALID)].reset_index(drop=True)
sw_base = run_sweep(CO_EV, "base", co_maps, [("real", ALPHA_STAR), ("random", ALPHA_STAR)], CP, "G_base")
sw_A = run_sweep(CO_EV, "A_only", co_maps, [("real", ALPHA_STAR)], {"b": "q_b"}, "G_Aonly")

co_g = score(sw_base.rename(columns={"db": "dcause", "dc": "deffect"}),
             co_sc.rename(columns={"db_see": "dcause_see", "dc_see": "deffect_see"}))
co_g.to_csv(f"{OUT}/G_scenario_scores.csv", index=False)
co_summ = summarise(co_g); co_summ.to_csv(f"{OUT}/G_summary.csv", index=False)
display(co_summ.groupby("kind")[["works_rate", "D_median"]].mean().round(2))

# explaining-away interaction under the edit (real direction)
nb = co_ref.groupby("scenario_id")[["b_base", "b_A_only"]].mean()
eb = sw_base[sw_base.kind == "real"].groupby(["scenario_id", "layer", "position"]).raw_b.mean().rename("b_editC")
ea = sw_A.groupby(["scenario_id", "layer", "position"]).raw_b.mean().rename("b_editC_A")
inter = pd.concat([eb, ea], axis=1).reset_index().merge(nb, left_on="scenario_id", right_index=True)
inter["EA_interaction_edit"] = (inter.b_editC - inter.b_editC_A) - (inter.b_base - inter.b_A_only)
inter = inter.merge(co_g[co_g.kind == "real"][["scenario_id", "layer", "position", "works"]], on=["scenario_id", "layer", "position"])
inter = inter.merge(co_sc[["EA_interaction_text"]], left_on="scenario_id", right_index=True)
inter.to_csv(f"{OUT}/G_interaction.csv", index=False)
irows = []
for (L, pos), grp in inter[inter.works].groupby(["layer", "position"]):
    lo, hi = boot_median(grp.EA_interaction_edit)
    irows.append(dict(layer=L, position=pos, n=len(grp), EA_edit=grp.EA_interaction_edit.median(), ci_lo=lo, ci_hi=hi,
                      EA_text=grp.EA_interaction_text.median()))
g_int = pd.DataFrame(irows); g_int.to_csv(f"{OUT}/G_interaction_summary.csv", index=False)
display(g_int.round(2))
print(f"Text explaining-away interaction, median over valid scenarios: {co_sc.loc[CO_VALID, 'EA_interaction_text'].median():.2f}")

fig, ax = plt.subplots(figsize=(4.5, 5))
im = heatmap(ax, co_summ[co_summ.kind == "real"], f"Colliders: edit C (alpha={ALPHA_STAR})")
ax.set_ylabel("layer"); fig.colorbar(im, ax=ax, label="median D (0 = seeing, 1 = doing)")
plt.savefig(f"{OUT}/G_collider_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()